# Quarterly Water Mask Composite Export (2015-2025)

## Overview
This notebook collects quarterly water mask composites for the Padma River study area using:
- **Sentinel-2 (Primary)**: Optical imagery with MNDWI for water detection
- **Sentinel-1 (Fallback)**: SAR imagery when optical data is insufficient

## Quarters Definition
- **Q1**: January - March (Dry Season)
- **Q2**: April - June (Pre-Monsoon)
- **Q3**: July - September (Monsoon Peak)
- **Q4**: October - December (Post-Monsoon)

## Workflow
1. Initialize Earth Engine
2. Define study area and parameters
3. Process quarterly composites for each year
4. Export water masks as GeoTIFF files
5. Generate summary statistics

In [ ]:
# Initialize Earth Engine
import ee
import geemap
import os
import time
import numpy as np
import pandas as pd

try:
    ee.Initialize(project='river-468515')  # Replace with your project ID
    print("✅ Earth Engine initialized successfully!")
except:
    ee.Authenticate()
    ee.Initialize(project='river-468515')
    print("✅ Earth Engine authenticated and initialized!")

In [ ]:
# =================================================================
# CONFIGURATION
# =================================================================

# Study Area: Padma River, Bangladesh
study_area = ee.Geometry.Polygon([
    [88.75677098777692, 24.00347653384235],
    [90.59972753074567, 23.152644569433964],
    [90.52556981590192, 23.54602121781216],
    [88.74480244337559, 24.334092333430064],
    [88.75677098777692, 24.00347653384235]
])

# Time Range
START_YEAR = 2015  # Sentinel-2 availability
END_YEAR = 2025

# Output Directory
out_dir = "./Quarterly_WaterMasks_2015-2025"
os.makedirs(out_dir, exist_ok=True)

# Processing Parameters
WATER_THRESHOLD = 0  # MNDWI threshold for water detection
CLOUD_COVER_MAX = 50  # Maximum cloud cover percentage
EXPORT_SCALE = 60  # Export resolution in meters
MAX_S2_IMAGES = 25  # Limit Sentinel-2 images per quarter
MAX_S1_IMAGES = 20  # Limit Sentinel-1 images per quarter

# Quarter Definitions
QUARTERS = {
    'Q1': {'months': [1, 2, 3], 'name': 'Jan-Mar'},
    'Q2': {'months': [4, 5, 6], 'name': 'Apr-Jun'},
    'Q3': {'months': [7, 8, 9], 'name': 'Jul-Sep'},
    'Q4': {'months': [10, 11, 12], 'name': 'Oct-Dec'}
}

print("="*70)
print("Quarterly Water Mask Export Configuration")
print("="*70)
print(f"Study Period      : {START_YEAR} - {END_YEAR}")
print(f"Total Quarters    : {(END_YEAR - START_YEAR + 1) * 4}")
print(f"Cloud Cover Max   : {CLOUD_COVER_MAX}%")
print(f"Export Scale      : {EXPORT_SCALE}m")
print(f"Image Limits      : S2={MAX_S2_IMAGES}, S1={MAX_S1_IMAGES}")
print("="*70)

In [ ]:
# =================================================================
# HELPER FUNCTIONS
# =================================================================

def mask_sentinel2_clouds(image):
    """Mask clouds in Sentinel-2 imagery using QA60 band."""
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10  # Bit 10: Opaque clouds
    cirrus_bit_mask = 1 << 11  # Bit 11: Cirrus clouds
    
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )
    return image.updateMask(mask).divide(10000)  # Scale to reflectance


def add_sentinel1_ratio(image):
    """Add VV/VH ratio band for Sentinel-1 SAR water detection."""
    vv = image.select('VV')
    vh = image.select('VH')
    ratio = vv.divide(vh).rename('VV_VH_ratio')
    return image.addBands(ratio)


def calculate_water_area_km2(water_mask, geometry, scale=30):
    """Calculate water area in square kilometers."""
    pixel_area = water_mask.multiply(ee.Image.pixelArea())
    water_area = pixel_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e10,
        bestEffort=True
    )
    area_km2 = ee.Number(water_area.get('water', 0)).divide(1e6)
    return area_km2.getInfo()


print("✅ Helper functions defined successfully!")

In [ ]:
# =================================================================
# MAIN PROCESSING FUNCTION
# =================================================================

def export_quarterly_water_mask(year, quarter, quarter_months, quarter_name):
    """
    Export quarterly water mask using Sentinel-2 (primary) and Sentinel-1 (fallback).
    
    Parameters:
    -----------
    year : int
        Year to process
    quarter : str
        Quarter identifier (Q1, Q2, Q3, Q4)
    quarter_months : list
        List of months in the quarter
    quarter_name : str
        Human-readable quarter name
    
    Returns:
    --------
    dict : Processing results including status, area, and image counts
    """
    try:
        # Define date range
        start_month = quarter_months[0]
        end_month = quarter_months[-1]
        start_date = ee.Date.fromYMD(year, start_month, 1)
        end_date = ee.Date.fromYMD(year, end_month, 1).advance(1, 'month')
        
        # ========================================================
        # SENTINEL-2 Processing (Primary - Optical)
        # ========================================================
        s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
            .filterDate(start_date, end_date) \
            .filterBounds(study_area) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_COVER_MAX)) \
            .sort('CLOUDY_PIXEL_PERCENTAGE') \
            .limit(MAX_S2_IMAGES) \
            .map(mask_sentinel2_clouds) \
            .select(['B3', 'B11'], ['Green', 'SWIR1'])
        
        s2_count = s2_collection.size().getInfo()
        
        # ========================================================
        # SENTINEL-1 Processing (Fallback - SAR)
        # ========================================================
        s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
            .filterDate(start_date, end_date) \
            .filterBounds(study_area) \
            .filter(ee.Filter.eq('instrumentMode', 'IW')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
            .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
            .sort('system:time_start', False) \
            .limit(MAX_S1_IMAGES) \
            .select(['VV', 'VH']) \
            .map(add_sentinel1_ratio)
        
        s1_count = s1_collection.size().getInfo()
        
        # Check if data is available
        if s2_count == 0 and s1_count == 0:
            return {
                'year': year, 'quarter': quarter,
                'water_area_km2': 0, 'filename': 'N/A',
                'status': 'no_images', 'image_count': 0,
                's2_count': 0, 's1_count': 0
            }
        
        # ========================================================
        # Water Detection
        # ========================================================
        water_mask = None
        method_used = ''
        
        if s2_count > 0:
            # Sentinel-2 MNDWI (Modified Normalized Difference Water Index)
            s2_composite = s2_collection.median().clip(study_area)
            mndwi = s2_composite.normalizedDifference(['Green', 'SWIR1'])
            water_mask = mndwi.gt(WATER_THRESHOLD).rename('water').toByte()
            method_used = 'S2_MNDWI'
        elif s1_count > 0:
            # Sentinel-1 SAR (VV polarization threshold)
            s1_composite = s1_collection.median().clip(study_area)
            vv = s1_composite.select('VV')
            water_mask = vv.lt(-16).rename('water').toByte()  # Water has low backscatter
            method_used = 'S1_VV'
        
        if water_mask is None:
            return {
                'year': year, 'quarter': quarter,
                'water_area_km2': 0, 'filename': 'N/A',
                'status': 'processing_failed', 'image_count': 0,
                's2_count': s2_count, 's1_count': s1_count
            }
        
        # Calculate water area
        area_km2 = calculate_water_area_km2(water_mask, study_area)
        
        if area_km2 == 0:
            return {
                'year': year, 'quarter': quarter,
                'water_area_km2': 0, 'filename': 'N/A',
                'status': 'no_water', 'image_count': s2_count + s1_count,
                's2_count': s2_count, 's1_count': s1_count
            }
        
        # Export water mask
        out_tif = os.path.join(out_dir, f"water_mask_{year:04d}_{quarter}.tif")
        
        try:
            geemap.ee_export_image(
                water_mask.selfMask(),
                filename=out_tif,
                scale=EXPORT_SCALE,
                region=study_area,
                file_per_band=False,
                crs='EPSG:4326',
                timeout=300
            )
            
            return {
                'year': year, 'quarter': quarter,
                'water_area_km2': round(area_km2, 2),
                'filename': os.path.basename(out_tif),
                'status': 'success',
                'image_count': s2_count + s1_count,
                's2_count': s2_count, 's1_count': s1_count,
                'method': method_used
            }
        except Exception as e:
            return {
                'year': year, 'quarter': quarter,
                'water_area_km2': round(area_km2, 2),
                'filename': 'N/A',
                'status': 'export_failed',
                'image_count': s2_count + s1_count,
                's2_count': s2_count, 's1_count': s1_count
            }
    
    except Exception as e:
        return {
            'year': year, 'quarter': quarter,
            'water_area_km2': 0, 'filename': 'N/A',
            'status': f'error: {str(e)[:30]}',
            'image_count': 0, 's2_count': 0, 's1_count': 0
        }


print("✅ Main processing function defined!")

In [ ]:
# =================================================================
# BATCH PROCESSING
# =================================================================

print("="*70)
print("STARTING QUARTERLY WATER MASK EXPORT")
print("="*70)

# Generate all quarter periods
all_periods = []
for year in range(START_YEAR, END_YEAR + 1):
    for quarter_id, quarter_info in QUARTERS.items():
        all_periods.append({
            'year': year,
            'quarter': quarter_id,
            'quarter_name': quarter_info['name'],
            'months': quarter_info['months']
        })

total_periods = len(all_periods)
print(f"Total quarters to process: {total_periods}\n")

# Process each period
results = []
export_summary = {
    'successful': 0,
    'failed': 0,
    'total_area_km2': 0,
    's2_primary': 0,
    's1_fallback': 0
}

for i, period_info in enumerate(all_periods):
    year = period_info['year']
    quarter = period_info['quarter']
    label = f"{year}-{quarter} ({period_info['quarter_name']})"
    
    print(f"[{i+1}/{total_periods}] Processing {label}...", end='', flush=True)
    
    result = export_quarterly_water_mask(
        year, quarter, period_info['months'], period_info['quarter_name']
    )
    results.append(result)
    
    if result['status'] == 'success':
        export_summary['successful'] += 1
        export_summary['total_area_km2'] += result['water_area_km2']
        
        if result.get('method') == 'S2_MNDWI':
            export_summary['s2_primary'] += 1
        elif result.get('method') == 'S1_VV':
            export_summary['s1_fallback'] += 1
        
        print(f" ✅ {result['method']} ({result['water_area_km2']} km², S2:{result['s2_count']} S1:{result['s1_count']})")
    else:
        export_summary['failed'] += 1
        print(f" ❌ {result['status']}")
    
    time.sleep(2)  # Rate limiting

# Save results to CSV
df_results = pd.DataFrame(results)
csv_path = os.path.join(out_dir, "quarterly_export_summary.csv")
df_results.to_csv(csv_path, index=False)

print("\n" + "="*70)
print("EXPORT COMPLETE")
print("="*70)
print(f"✅ Successful exports: {export_summary['successful']}/{total_periods}")
print(f"❌ Failed exports    : {export_summary['failed']}")
print(f"\n📊 Method Distribution:")
print(f"   • Sentinel-2 (MNDWI): {export_summary['s2_primary']}")
print(f"   • Sentinel-1 (SAR) : {export_summary['s1_fallback']}")

if export_summary['successful'] > 0:
    avg_area = export_summary['total_area_km2'] / export_summary['successful']
    print(f"\n💧 Average Water Area: {avg_area:.2f} km²")

print(f"\n📁 Output Directory: {out_dir}")
print(f"📄 Summary CSV: {csv_path}")
print("="*70)

## 📊 Data Summary and Visualization

Analyze the exported data and create visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load results
df = pd.read_csv(csv_path)
df_success = df[df['status'] == 'success'].copy()

# Plot 1: Water Area by Quarter Over Time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Temporal trend
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    df_q = df_success[df_success['quarter'] == q]
    ax1.plot(df_q['year'], df_q['water_area_km2'], marker='o', label=q, linewidth=2)

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Water Area (km²)', fontsize=12)
ax1.set_title('Quarterly Water Area Temporal Variation (2015-2025)', fontsize=14, fontweight='bold')
ax1.legend(title='Quarter')
ax1.grid(True, alpha=0.3)

# Seasonal pattern
quarter_avg = df_success.groupby('quarter')['water_area_km2'].mean().reindex(['Q1', 'Q2', 'Q3', 'Q4'])
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
ax2.bar(quarter_avg.index, quarter_avg.values, color=colors, alpha=0.8, edgecolor='black')
ax2.set_xlabel('Quarter', fontsize=12)
ax2.set_ylabel('Average Water Area (km²)', fontsize=12)
ax2.set_title('Average Water Area by Quarter', fontsize=14, fontweight='bold')
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'quarterly_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved!")

In [ ]:
# Statistical Summary
print("\n" + "="*70)
print("STATISTICAL SUMMARY")
print("="*70)

print("\n📊 Overall Statistics:")
print(df_success['water_area_km2'].describe())

print("\n📊 Quarterly Statistics:")
print(df_success.groupby('quarter')['water_area_km2'].describe())

print("\n📊 Image Count Statistics:")
print(f"Average Sentinel-2 images per quarter: {df_success['s2_count'].mean():.1f}")
print(f"Average Sentinel-1 images per quarter: {df_success['s1_count'].mean():.1f}")
print(f"Total images processed: {df_success['image_count'].sum()}")

print("\n" + "="*70)

## ✅ Completion

The quarterly water mask collection is complete! The exported GeoTIFF files are ready for:
1. Gap-filling analysis
2. Model training (ConvLSTM, Swin Transformer)
3. Statistical analysis
4. Temporal trend analysis

### Next Steps:
- Run `preprocessing/01_gap_filling_methods.ipynb` to fill any data gaps
- Proceed to model training notebooks in the `models/` directory
- Generate comprehensive statistics using `results/01_statistical_analysis.ipynb`